# Mushroom Classification with ViT


## 1. Налаштування середовища

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не знайдено! Перейдіть в Runtime → Change runtime type → GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive підключено")

In [ ]:

import os
os.chdir('/content')

if os.path.exists('Applied_Mashroomatics'):
    os.chdir('Applied_Mashroomatics')
    !git pull
else:
    !git clone https://github.com/konstanta-asya/Applied_Mashroomatics.git
    os.chdir('Applied_Mashroomatics')

print(f"Робоча директорія: {os.getcwd()}")

In [ ]:
!pip install -q tqdm timm

In [ ]:
import os

!rm -rf data/raw
!mkdir -p data/raw

DATA_PATH = "/content/drive/MyDrive/mushroom_data"  # або "/content/drive/MyDrive/raw"

!ln -s {DATA_PATH}/DF20M data/raw/DF20M
!ln -s {DATA_PATH}/DF20M-metadata data/raw/DF20M-metadata

if os.path.exists('data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv'):
    print("Дані підключено успішно")
    !ls data/raw/DF20M-metadata/
else:
    print("Дані не знайдено! Перевірте шлях до папки на Google Drive")
    print(f"   Очікується: {DATA_PATH}/DF20M та {DATA_PATH}/DF20M-metadata")

## 2. Тренування моделі

**Варіанти швидкості:**
- 🚀 **Найшвидше** (~20-30 хв): `model=vit_small`, `freeze_backbone=True`
- ⚡ **Швидке** (~40-60 хв): `model=vit_base`, `freeze_backbone=True`
- 🎯 **Висока точність** (~1.5-2 год/epoch): `model=vit_base`, `freeze_backbone=False`

**Оптимізації:**
- Mixed precision (AMP) - ~1.5-2x швидше
- torch.compile - додаткове прискорення
- **RESUME** - продовжує з останнього checkpoint якщо Colab відключився!

In [ ]:
# === HIGH ACCURACY SETTINGS ===
MODEL = "vit_base"        # Більша модель = краща точність
BATCH_SIZE = 32           # Менший batch для стабільності
EPOCHS = 15               # Більше epochs
LEARNING_RATE = 1e-4      # Нижчий lr для fine-tuning
FREEZE_BACKBONE = False   # Тренуємо ВСЕ
USE_COMPILE = True
RESUME = True             # Продовжити з checkpoint якщо є

freeze_flag = "--freeze_backbone" if FREEZE_BACKBONE else ""
compile_flag = "--compile" if USE_COMPILE else ""
resume_flag = f"--resume /content/drive/MyDrive/mushroom_checkpoints/latest_checkpoint.pth" if RESUME else ""

!python src/train_vit.py \
    --csv_path data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv \
    --data_dir data/raw/DF20M \
    --model {MODEL} \
    --batch_size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --lr {LEARNING_RATE} \
    --amp \
    {freeze_flag} \
    {compile_flag} \
    --num_workers 2 \
    --checkpoint_dir /content/drive/MyDrive/mushroom_checkpoints \
    {resume_flag}

## 3. Результати

In [ ]:
import os

checkpoint_dir = '/content/drive/MyDrive/mushroom_checkpoints'

if os.path.exists(checkpoint_dir):
    print("📁 Збережені файли:")
    for f in os.listdir(checkpoint_dir):
        size = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f"  - {f} ({size:.1f} MB)")
else:
    print("❌ Папка з чекпоінтами не знайдена")

In [ ]:

from google.colab import files
import os

best_model = '/content/drive/MyDrive/mushroom_checkpoints/best_model.pth'

if os.path.exists(best_model):
    files.download(best_model)
    print("✅ Модель завантажено")
else:
    print("❌ Файл best_model.pth не знайдено. Тренування ще не завершено?")

## 4. Тестування моделі

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
import sys
sys.path.insert(0, '/content/Applied_Mashroomatics')
from src.models.vit import create_vit_model, create_vit_small

checkpoint = torch.load('/content/drive/MyDrive/mushroom_checkpoints/best_model.pth')
num_classes = checkpoint['num_classes']
model_type = checkpoint.get('model_type', 'vit_base')  # Default to vit_base for old checkpoints

print(f"Model type: {model_type}")
print(f"Кількість класів: {num_classes}")
print(f"Точність на валідації: {checkpoint['val_acc']:.2f}%")

if model_type == 'vit_small':
    model = create_vit_small(num_classes=num_classes, pretrained=False)
else:
    model = create_vit_model(num_classes=num_classes, pretrained=False)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Модель завантажено і готова до використання")

In [ ]:
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# image_path = '/content/drive/MyDrive/your_mushroom.jpg'
# image = Image.open(image_path).convert('RGB')
# input_tensor = transform(image).unsqueeze(0)

# with torch.no_grad():
#     output = model(input_tensor)
#     predicted_class = output.argmax(1).item()
#     confidence = torch.softmax(output, dim=1)[0][predicted_class].item()

# print(f"Predicted class: {predicted_class}")
# print(f"Confidence: {confidence:.2%}")

